### Persistant memory

In [1]:
import operator
import os
import re
from typing import Annotated, List, Literal

import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send

load_dotenv()  

d:\d\NLP\agentic-ai-revision\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
C:\Users\Nour_\AppData\Local\Temp\ipykernel_56168\3455488566.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\d\NLP\agentic-ai-revision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
llm.invoke("hi").content

'Hello! How can I help you today?'

In [6]:
embeddings =  HuggingFaceEmbeddings()
embeddings

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 368.39it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:
#document ingestiona dn parsing

pdfs = [
    "project1/1706.03762v7.pdf",   # Attention Is All You Need (Transformer)
    "project1/2005.11401v4.pdf",   # Retrieval-Augmented Generation (RAG)
    "project1/2210.03629v3.pdf",   # ReAct
    "project1/2310.11511v1.pdf",   # Self-RAG
    "project1/2401.15884v3.pdf",   # Corrective RAG (CRAG)
]

documents= []
for pdf in pdfs:
    doc = PyPDFLoader(pdf).load()
    documents.extend(doc)
    print(f"{pdf}: {len(doc)} pages")
print(f"Total pages: {len(documents)}")


project1/1706.03762v7.pdf: 15 pages
project1/2005.11401v4.pdf: 19 pages
project1/2210.03629v3.pdf: 33 pages
project1/2310.11511v1.pdf: 30 pages
project1/2401.15884v3.pdf: 16 pages
Total pages: 113


In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splitter.split_documents(documents)
vs = Chroma.from_documents(chunks, embeddings)

In [20]:
@tool
def retrieve_tool(query:str):
    """Retrieve the information related to the query"""
    result = vs.as_retriever().invoke(query,k=2)
    serialized_docs = "\n\n".join(
        (f"source: {doc.metadata}\Content:{doc.page_content}")
        for doc in result
    )
    return serialized_docs , result

<>:6: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<>:6: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
C:\Users\Nour_\AppData\Local\Temp\ipykernel_56168\2999000064.py:6: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
  (f"source: {doc.metadata}\Content:{doc.page_content}")


In [22]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState , StateGraph

def query_or_response(state: MessagesState):
    """Generate tool call for retrieval or respond"""
    llm_with_tools = llm.bind_tools([retrieve_tool])
    response = llm_with_tools.invoke(state["messages"])
    return {"messages":{response}}